In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/anushkumar117/illegal/Valid.csv
/kaggle/input/datasets/anushkumar117/illegal/Train.csv
/kaggle/input/datasets/anushkumar117/illegal/Test.csv


In [2]:
!pip install transformers datasets pandas scikit-learn accelerate

import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ======================================================
# CHANGE ONLY THESE FOR ANY DATASET
# ======================================================
ID_COLUMN    = "id"        # 3-column dataset id/text/label
TEXT_COLUMN  = "text"      # sentiment = text | spam = email
LABEL_COLUMN = "label"

# OUTPUT LABELS
LABEL_0_NAME = "Negative"
LABEL_1_NAME = "Positive"

# For Spam Dataset Use:
# TEXT_COLUMN  = "email"
# LABEL_0_NAME = "Not Spam"
# LABEL_1_NAME = "Spam"

# ======================================================
# LOAD DATA
# ======================================================
train_df = pd.read_csv("/kaggle/input/datasets/anushkumar117/illegal/Train.csv")
test_df  = pd.read_csv("/kaggle/input/datasets/anushkumar117/illegal/Test.csv")
valid_df = pd.read_csv("/kaggle/input/datasets/anushkumar117/illegal/Valid.csv")

# ======================================================
# BASIC CHECKS
# ======================================================
train_df.drop_duplicates(inplace=True)
test_df.drop_duplicates(inplace=True)
valid_df.drop_duplicates(inplace=True)

# ======================================================
# CLEAN TEXT COLUMN
# ======================================================
train_df[TEXT_COLUMN] = train_df[TEXT_COLUMN].fillna("").astype(str)
test_df[TEXT_COLUMN]  = test_df[TEXT_COLUMN].fillna("").astype(str)
valid_df[TEXT_COLUMN] = valid_df[TEXT_COLUMN].fillna("").astype(str)

# ======================================================
# CONVERT TO DATASET
# ======================================================
train_dataset = Dataset.from_pandas(train_df)
test_dataset  = Dataset.from_pandas(test_df)
valid_dataset = Dataset.from_pandas(valid_df)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": valid_dataset,
    "test": test_dataset
})

# ======================================================
# TOKENIZER
# ======================================================
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(example):
    texts = [str(x) for x in example[TEXT_COLUMN]]

    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

tokenized_datasets = tokenized_datasets.remove_columns([TEXT_COLUMN])
tokenized_datasets = tokenized_datasets.rename_column(LABEL_COLUMN, "labels")

# remove id column from training tensors if exists
if ID_COLUMN in tokenized_datasets["train"].column_names:
    tokenized_datasets = tokenized_datasets.remove_columns([ID_COLUMN])

tokenized_datasets.set_format("torch")

# ======================================================
# MODEL
# ======================================================
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="no",
    logging_dir="./logs",
    logging_steps=200,
    dataloader_pin_memory=False
)

# ======================================================
# METRICS
# ======================================================
def compute_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary"
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# ======================================================
# TRAINER
# ======================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics
)

trainer.train()

trainer.evaluate()

# ======================================================
# SINGLE TEXT PREDICTION
# ======================================================
def predict(text):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    inputs = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class_id = torch.argmax(logits, dim=1).item()

    return LABEL_1_NAME if predicted_class_id == 1 else LABEL_0_NAME

print(predict("A beautiful film with emotional depth. The performances were powerful."))

print(predict("Worst movie ever made. Waste of time."))

user_review = input("Enter Text: ")
print(predict(user_review))

# ======================================================
# TEST PREDICTIONS + SUBMISSION FILE
# ======================================================
'''
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)

submission = pd.DataFrame({
    ID_COLUMN: test_df[ID_COLUMN],
    TEXT_COLUMN: test_df[TEXT_COLUMN],
    "predicted_label": predicted_labels
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")
print(submission.head())
'''
# ======================================================
# LABEL NAME SUBMISSION FILE
# ======================================================
label_map = {
    0: LABEL_0_NAME,
    1: LABEL_1_NAME
}

predicted_names = [label_map[x] for x in predicted_labels]

submission = pd.DataFrame({
    ID_COLUMN: test_df[ID_COLUMN],
    TEXT_COLUMN: test_df[TEXT_COLUMN],
    "predicted_label": predicted_names
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")
print(submission.head())

results = trainer.evaluate()
print(results)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

...

Enter Text:  I am happy after the movie


Positive


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


submission.csv created successfully!
    id                                               text  predicted_label
0  111  I always wrote this series off as being a comp...                0
1  112  1st watched 12/7/2002 - 3 out of 10(Dir-Steve ...                0
2  113  This movie was so poorly written and directed ...                0
3  114  The most interesting thing about Miryang (Secr...                1
4  115  when i first read about "berlin am meer" i did...                0


'\n# ======================================================\n# LABEL NAME SUBMISSION FILE\n# ======================================================\nlabel_map = {\n    0: LABEL_0_NAME,\n    1: LABEL_1_NAME\n}\n\npredicted_names = [label_map[x] for x in predicted_labels]\n\nsubmission = pd.DataFrame({\n    ID_COLUMN: test_df[ID_COLUMN],\n    TEXT_COLUMN: test_df[TEXT_COLUMN],\n    "predicted_label": predicted_names\n})\n\nsubmission.to_csv("submission.csv", index=False)\n\nprint("submission.csv created successfully!")\nprint(submission.head())\n\nresults = trainer.evaluate()\nprint(results)\n'

In [4]:
label_map = {
    0: LABEL_0_NAME,
    1: LABEL_1_NAME
}

predicted_names = [label_map[x] for x in predicted_labels]

submission = pd.DataFrame({
    ID_COLUMN: test_df[ID_COLUMN],
    TEXT_COLUMN: test_df[TEXT_COLUMN],
    "predicted_label": predicted_names
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")
print(submission.head())

submission.csv created successfully!
    id                                               text predicted_label
0  111  I always wrote this series off as being a comp...        Negative
1  112  1st watched 12/7/2002 - 3 out of 10(Dir-Steve ...        Negative
2  113  This movie was so poorly written and directed ...        Negative
3  114  The most interesting thing about Miryang (Secr...        Positive
4  115  when i first read about "berlin am meer" i did...        Negative


In [3]:
results = trainer.evaluate()
print(results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.5769855380058289, 'eval_accuracy': 0.8938, 'eval_f1': 0.8958210712183637, 'eval_precision': 0.883855981416957, 'eval_recall': 0.9081145584725537, 'eval_runtime': 25.307, 'eval_samples_per_second': 197.574, 'eval_steps_per_second': 6.204, 'epoch': 2.0}
